# 1.5. Загрузка и интеграция данных из различных форматов (Лекция_5)

Способы соединения и совмещения данных: объединение таблиц, импорт из CSV/JSON/XML. Обработка ошибок и конфликтов форматов: кодировки, несовместимые типы, пропущенные значения. Инструменты интеграции: Python (pandas), SQL (LOAD DATA), ETL-платформы (например, Pentaho, Talend).



## Задание №1. Соединение данных из нескольких источников (SQL JOIN)

Задача:Есть две таблицы: одна содержит информацию о сотрудниках (Employees), вторая — о департаментах (Departments). Каждая запись в таблице сотрудников ссылается на соответствующий департамент. Объедините обе таблицы, используя SQL JOIN, чтобы получить полную информацию обо всех сотрудниках и их департаментах.

In [41]:
import sqlite3

# Создаем временную базу данных
conn = sqlite3.connect(':memory:')
cur = conn.cursor()

# Создаем таблицы
cur.execute("""CREATE TABLE Employees(employee_id INTEGER PRIMARY KEY, name TEXT NOT NULL, department_id INTEGER NOT NULL)""")

cur.execute("""CREATE TABLE Departments(department_id INTEGER PRIMARY KEY, department_name TEXT NOT NULL)""")

# Вставляем данные
employees_data = [(1, 'John', 1), (2, 'Jane', 2), (3, 'Kane', 3), (4, 'Kate', 3), (5, 'Mary', 2), (6, 'Bill', 1)]
departments_data = [(1, 'IT'), (2, 'Sales'), (3, 'Support'), (4, 'Security')]

cur.executemany("INSERT INTO Employees (employee_id, name, department_id) VALUES (?, ?, ?)", employees_data)
cur.executemany("INSERT INTO Departments (department_id, department_name) VALUES (?, ?)", departments_data)

# Запрашиваем объединённые данные
cur.execute("""
SELECT e.employee_id, e.name, d.department_name
FROM Departments AS d
INNER JOIN Employees AS e ON d.department_id = e.department_id
ORDER BY e.employee_id
""")

# Выводим результат
results = cur.fetchall()
for result in results:
    print(result)

# Закрываем подключение
conn.close()

(1, 'John', 'IT')
(2, 'Jane', 'Sales')
(3, 'Kane', 'Support')
(4, 'Kate', 'Support')
(5, 'Mary', 'Sales')
(6, 'Bill', 'IT')


---------------------------------------------------------------

## Задание №2. Слияние данных из CSV и JSON

In [42]:
import pandas as pd
import json

# Шаг 1: Создаем файлы с данными

# Создаем CSV-файл с данными о складах
with open('warehouses.csv', 'w') as f:
  f.write('warehouse_id,location\n1,Moscow\n2,Saint Petersburg\n3,Voronezh\n4,Krasnodar\n5,Paris\n6,Tokyo\n')

# Создаем JSON-файл с данными о товарах
with open('products.json', 'w') as f:
    json.dump([
      {"product_id": 1, "warehouse_id": 1, "product_name": "Chair"},
      {"product_id": 2, "warehouse_id": 2, "product_name": "Table"},
      {"product_id": 3, "warehouse_id": 1, "product_name": "Desk"},
      {"product_id": 4, "warehouse_id": 3, "product_name": "Computer"},
      {"product_id": 5, "warehouse_id": 6, "product_name": "Laptop"},
      {"product_id": 6, "warehouse_id": 5, "product_name": "Headphones"},
      {"product_id": 7, "warehouse_id": 4, "product_name": "Lamp"},
      {"product_id": 8, "warehouse_id": 2, "product_name": "Sofa"},
      {"product_id": 9, "warehouse_id": 1, "product_name": "Chair"}
    ], f)

# Шаг 2: Загружаем данные из CSV и JSON
warehouses = pd.read_csv('warehouses.csv')

# Загружаем данные из JSON
with open('products.json', 'r') as f:
    products = json.load(f)

# Приводим данные к DataFrame
products_df = pd.DataFrame.from_records(products)

# Шаг 3: Производим слияние
merged_data = warehouses.merge(products_df, on='warehouse_id', how='inner')

# Шаг 4: Выводим результат
print(merged_data)

   warehouse_id          location  product_id product_name
0             1            Moscow           1        Chair
1             1            Moscow           3         Desk
2             1            Moscow           9        Chair
3             2  Saint Petersburg           2        Table
4             2  Saint Petersburg           8         Sofa
5             3          Voronezh           4     Computer
6             4         Krasnodar           7         Lamp
7             5             Paris           6   Headphones
8             6             Tokyo           5       Laptop


---------------------------------------------------------------

## Задание №3: Добавление новых записей и фильтрация по условию

Создать временные базы данных, заполнить их необходимыми таблицами и добавить дополнительные записи сотрудников и отделов. Затем выбрать всех сотрудников, работающих в определенном отделе, используя условие WHERE.


In [43]:
import sqlite3

# Создаем временную базу данных
conn = sqlite3.connect(':memory:')
cur = conn.cursor()

# Создаем таблицы
cur.execute("""
CREATE TABLE Employees(
    employee_id INT PRIMARY KEY,
    name TEXT,
    salary REAL,
    department_id INT
)
""")

cur.execute("""
CREATE TABLE Departments(
    department_id INT PRIMARY KEY,
    department_name TEXT
)
""")

# Вставляем начальные данные
employees_data = [
    (1, 'Иван Петров', 50000, 1),
    (2, 'Анна Смирнова', 70000, 2),
    (3, 'Сергей Кузнецов', 60000, 3),
    (4, 'Екатерина Морозова', 75000, 3),
    (5, 'Мария Ковалева', 65000, 2),
    (6, 'Борис Соколов', 55000, 1),
]
departments_data = [
    (1, 'Отдел IT'),
    (2, 'Продажи'),
    (3, 'Support'),
    (4, 'Security'),
]

cur.executemany("INSERT INTO Employees (employee_id, name, salary, department_id) VALUES (?, ?, ?, ?)", employees_data)
cur.executemany("INSERT INTO Departments (department_id, department_name) VALUES (?, ?)", departments_data)

# Добавляем новые записи
new_employees = [
    (13, 'Михаил Иванов', 80000, 1),
    (14, 'Ольга Васильева', 60000, 2),
    (15, 'Иван Михайлов', 380000, 3),
    (16, 'Василий Ольгов', 360000, 3),
    (17, 'Иванка Михалко', 480000, 4),
    (18, 'Василиса Олегович', 460000, 4)
]
cur.executemany("INSERT INTO Employees VALUES (?,?,?,?)", new_employees)

# Запрашиваем всех сотрудников одного из отделов
cur.execute("""
SELECT employee_id, name, salary, department_id
FROM Employees
WHERE department_id = ?
ORDER BY employee_id
""", (3,))

# Выводим результат
results = cur.fetchall()
for row in results:
    print(row)

# Закрываем соединение
conn.close()

(3, 'Сергей Кузнецов', 60000.0, 3)
(4, 'Екатерина Морозова', 75000.0, 3)
(15, 'Иван Михайлов', 380000.0, 3)
(16, 'Василий Ольгов', 360000.0, 3)


------------------------------------------------------------

## Задание №4: Обновление зарплаты сотрудника и группировка по отделам

Создать таблицу сотрудников и департаментов, обновить зарплату конкретного сотрудника, вывести среднее значение зарплат по каждому департаменту.


In [44]:
import sqlite3

# Создаем временную базу данных
conn = sqlite3.connect(':memory:')
cur = conn.cursor()

# Создаем таблицы
cur.execute("""
  CREATE TABLE Employees(
      employee_id INTEGER PRIMARY KEY,
      name TEXT NOT NULL,
      salary REAL NOT NULL,
      department_id INTEGER NOT NULL
  )
""")

cur.execute("""
CREATE TABLE Departments(
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL
)
""")

# Вставляем данные
employees_data = [
    (1, 'Игорь Кузнецов', 50000, 1),
    (2, 'Светлана Петрова', 70000, 2),
    (3, 'Дмитрий Сергеев', 60000, 1),
    (4, 'Елена Иванова', 80000, 2)
]
departments_data = [
    (1, 'Финансовый отдел'),
    (2, 'Маркетинг')
]

cur.executemany("INSERT INTO Employees (employee_id, name, salary, department_id) VALUES (?, ?, ?, ?)", employees_data)
cur.executemany("INSERT INTO Departments (department_id, department_name) VALUES (?, ?)", departments_data)

# Выбираем среднюю зарплату по каждому отделу ДО ПОВЫШЕНИЯ
cur.execute("""
  SELECT d.department_name, AVG(e.salary)
  FROM Departments d
  JOIN Employees e ON e.department_id = d.department_id
  GROUP BY d.department_name, d.department_id
  ORDER BY d.department_id
""")

# Выводим результат
print('Средняя зарплата по отделам ДО ПОВЫШЕНИЯ')
results = cur.fetchall()
for row in results:
    print(f'Средняя зарплата {row[0]} — {round(row[1], 2)} рублей.')

# Повышаем зарплату сотруднику на 10%
cur.execute("UPDATE Employees SET salary = salary * 1.25 WHERE employee_id=4")

# Выбираем среднюю зарплату по каждому отделу ПОСЛЕ ПОВЫШЕНИЯ
cur.execute("""
  SELECT d.department_name, AVG(e.salary)
  FROM Departments d
  JOIN Employees e ON e.department_id = d.department_id
  GROUP BY d.department_name, d.department_id
  ORDER BY d.department_id
""")

# Выводим результат
print('\n');
print('Средняя зарплата по отделам ПОСЛЕ ПОВЫШЕНИЯ')
results = cur.fetchall()
for row in results:
    print(f'Средняя зарплата {row[0]} — {round(row[1], 2)} рублей.')

# Закрываем соединение
conn.close()

Средняя зарплата по отделам ДО ПОВЫШЕНИЯ
Средняя зарплата Финансовый отдел — 55000.0 рублей.
Средняя зарплата Маркетинг — 75000.0 рублей.


Средняя зарплата по отделам ПОСЛЕ ПОВЫШЕНИЯ
Средняя зарплата Финансовый отдел — 55000.0 рублей.
Средняя зарплата Маркетинг — 85000.0 рублей.


-----------------------------------------------------------------------

## Задание №5: Удаление сотрудника и получение итоговых сумм по отделам

Создай две таблицы: сотрудники и отделы. Удали одного сотрудника и выведи общую сумму зарплат каждого отдела после удаления.

In [45]:
import sqlite3

# Создаем временную базу данных
conn = sqlite3.connect(':memory:')
cur = conn.cursor()

# Создаем таблицы
cur.execute("""
CREATE TABLE Employees(
    employee_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    salary REAL NOT NULL,
    department_id INTEGER NOT NULL
)
""")

cur.execute("""
CREATE TABLE Departments(
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL
)
""")

# Вставляем данные
employees_data = [
    (1, 'Алексей Семенов', 50000, 1),
    (2, 'Евгений Александров', 70000, 2),
    (3, 'Владимир Новиков', 60000, 1),
    (4, 'Дарья Андреева', 80000, 2)
]
departments_data = [
    (1, 'HR-отдел'),
    (2, 'Логистика')
]

cur.executemany("INSERT INTO Employees (employee_id, name, salary, department_id) VALUES (?, ?, ?, ?)", employees_data)
cur.executemany("INSERT INTO Departments (department_id, department_name) VALUES (?, ?)", departments_data)

# Получаем итоговую сумму зарплат по каждому отделу ДО УДАЛЕНИЯ
cur.execute("""
SELECT d.department_name, SUM(e.salary)
FROM Employees e
INNER JOIN Departments d USING (department_id)
GROUP BY d.department_id, d.department_name
ORDER BY d.department_id
""")

# Выводим результат
print('Сумма зарплат по отделам ДО УДАЛЕНИЯ')
results = cur.fetchall()
for row in results:
    print(f'Общая сумма зарплат в {row[0]} — {round(row[1], 2)} рублей.')

# Удаляем сотрудника
cur.execute("DELETE FROM Employees WHERE employee_id='3'")

# Получаем итоговую сумму зарплат по каждому отделу ПОСЛЕ УДАЛЕНИЯ
cur.execute("""
SELECT d.department_name, SUM(e.salary)
FROM Employees e
INNER JOIN Departments d USING (department_id)
GROUP BY d.department_id, d.department_name
ORDER BY d.department_id
""")

# Выводим результат
print('\n');
print('Сумма зарплат по отделам ПОСЛЕ УДАЛЕНИЯ')
results = cur.fetchall()
for row in results:
    print(f'Общая сумма зарплат в {row[0]} — {round(row[1], 2)} рублей.')

# Закрываем соединение
conn.close()

Сумма зарплат по отделам ДО УДАЛЕНИЯ
Общая сумма зарплат в HR-отдел — 110000.0 рублей.
Общая сумма зарплат в Логистика — 150000.0 рублей.


Сумма зарплат по отделам ПОСЛЕ УДАЛЕНИЯ
Общая сумма зарплат в HR-отдел — 50000.0 рублей.
Общая сумма зарплат в Логистика — 150000.0 рублей.
